# บทที่ 3 — จาก "mask" ไปสู่ "ป้อมต้องหมุนไปทางไหน"

🎯 เป้าหมาย: จาก mask ขาว-ดำ (บทที่แล้ว) หาตำแหน่งวัตถุ แล้วคำนวณว่าป้อมควรหมุนซ้าย/ขวากี่องศา
— บทนี้คือ**หัวใจของทั้งโปรเจค** เข้าใจบทนี้ = เข้าใจว่าป้อมยิงเล็งเป้าได้ยังไง

## 1. Contour คืออะไร — มาดูตัวอย่างง่ายๆ ก่อน

**Contour** = "เส้นขอบ" ของก้อนสีขาวใน mask พูดง่ายๆ คือโครงร่างรอบๆ วัตถุที่เจอ
ลองสร้าง mask ง่ายๆ (วงกลม 1 วง) แล้วดูว่า OpenCV หาเส้นขอบของมันยังไง

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Leelawadee UI"   # ฟอนต์นี้รองรับภาษาไทย กันตัวอักษรไทยในกราฟกลายเป็นกล่องว่าง

simple_mask = np.zeros((200, 200), np.uint8)
cv2.circle(simple_mask, (100, 80), 40, 255, -1)   # วาดวงกลมขาวลงบนพื้นดำ

# cv2.findContours หาเส้นขอบของก้อนขาวทั้งหมดใน mask
# RETR_EXTERNAL = เอาแค่เส้นขอบนอกสุด (ไม่สนใจรูข้างในถ้ามี)
# CHAIN_APPROX_SIMPLE = เก็บแค่จุดหักมุม ประหยัดหน่วยความจำ
contours, _ = cv2.findContours(simple_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print("จำนวน contour ที่เจอ:", len(contours))
print("จำนวนจุดในเส้นขอบวงกลมนี้:", len(contours[0]))

drawn_image = cv2.cvtColor(simple_mask, cv2.COLOR_GRAY2BGR)
cv2.drawContours(drawn_image, contours, -1, (0, 0, 255), 2)   # วาดเส้นขอบสีแดงทับ
plt.imshow(cv2.cvtColor(drawn_image, cv2.COLOR_BGR2RGB))
plt.title("เส้นขอบ (contour) สีแดง รอบก้อนขาว")
plt.show()

## 2. จาก contour หา "กรอบสี่เหลี่ยม" และ "จุดกึ่งกลาง"

`cv2.boundingRect(contour)` คืนค่า `(x, y, w, h)` = กรอบสี่เหลี่ยมที่เล็กที่สุดที่ครอบ contour นั้นพอดี
จากกรอบนี้ จุดกึ่งกลางของวัตถุ = `(x + w/2, y + h/2)`

In [ ]:
x, y, w, h = cv2.boundingRect(contours[0])
cx, cy = x + w // 2, y + h // 2

print(f"กรอบสี่เหลี่ยม: x={x}, y={y}, กว้าง={w}, สูง={h}")
print(f"จุดกึ่งกลางวัตถุ: ({cx}, {cy})")

cv2.rectangle(drawn_image, (x, y), (x + w, y + h), (0, 255, 0), 2)
cv2.circle(drawn_image, (cx, cy), 4, (0, 255, 0), -1)
plt.imshow(cv2.cvtColor(drawn_image, cv2.COLOR_BGR2RGB))
plt.title("กรอบเขียว + จุดกึ่งกลาง")
plt.show()

## 3. ใช้ของจริง: หา mask แล้วกรอง noise ด้วย `contourArea`

ในภาพจริง มักมี noise เป็นจุดขาวเล็กๆ กระจาย (หลาย contour) เราเลยต้อง:
1. หา contour ทั้งหมด
2. เลือกอันที่ **ใหญ่ที่สุด** (สมมติว่าวัตถุจริงต้องใหญ่กว่า noise)
3. กรองทิ้งถ้าพื้นที่เล็กเกินไป (กันกรณีไม่มีวัตถุจริงอยู่เลย แต่ noise ดันใหญ่พอดี)

In [ ]:
def _placeholder_scene():
    img = np.full((480, 640, 3), (235, 230, 220), np.uint8)
    cv2.rectangle(img, (0, 340), (640, 480), (170, 140, 90), -1)
    for x, y, r, color in [(160, 300, 55, (40, 170, 40)), (330, 290, 40, (60, 110, 150)),
                           (480, 280, 28, (130, 130, 130))]:
        cv2.circle(img, (x, y), r, color, -1)
    return img


def open_camera_or_placeholder(index=0):
    # อธิบายละเอียดแล้วในบทที่ 1
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW)
    if cap.isOpened():
        ok, frame = cap.read()
        cap.release()
        if ok:
            print(f"✅ ถ่ายจากกล้องจริงสำเร็จ! (index {index})")
            return frame
    print("⚠️ ไม่พบกล้อง — ใช้ภาพจำลองสนามแทน")
    return _placeholder_scene()


MIN_AREA = 800   # พื้นที่ (px²) ขั้นต่ำที่จะนับว่าเป็นวัตถุจริง ไม่ใช่ noise

def find_green_object(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([35, 80, 60]), np.array([85, 255, 255]))   # ช่วงสีเขียว จากบทที่แล้ว
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    largest_blob = max(contours, key=cv2.contourArea)   # max(...) เทียบทุกก้อนด้วยพื้นที่ เอาอันใหญ่สุด
    if cv2.contourArea(largest_blob) < MIN_AREA:
        return None
    x, y, w, h = cv2.boundingRect(largest_blob)
    return (x + w // 2, y + h // 2, w, h)


frame = open_camera_or_placeholder()
found_object = find_green_object(frame)

# กรณีถ่ายจากกล้องจริงแต่ไม่มีอะไรสีเขียวอยู่หน้ากล้อง (หรือแสงไม่เข้าเงื่อนไข) จะหาไม่เจอ
# — นี่คือสถานการณ์จริงที่เกิดได้เสมอ ในโปรเจคจริงป้อมจะ "กวาดหา" ต่อ (ดูโค้ดจริงท้ายบท)
# ที่นี่เราขอสลับไปใช้ภาพจำลองแทน เพื่อให้เรียนขั้นตอนต่อไปได้แน่นอน
if found_object is None:
    print("⚠️ ไม่เจอวัตถุสีเขียวในภาพนี้ (ลองเอาของสีเขียวมาส่องกล้อง หรือปรับช่วงสีจากบทที่ 2)")
    print("   สลับไปใช้ภาพจำลองสนามแทน เพื่อให้ทำบทเรียนต่อได้")
    frame = _placeholder_scene()
    found_object = find_green_object(frame)

print("ผลลัพธ์ (cx, cy, w, h):", found_object)

## 4. คำนวณ "error" — ป้อมต้องหมุนไปทางไหน กี่ px?

หลักการง่ายมาก: เทียบตำแหน่งวัตถุ (`cx`) กับ **จุดกึ่งกลางภาพ**
- error > 0 → วัตถุอยู่ **ขวา** ของกลางภาพ → ป้อมต้องหมุนขวา
- error < 0 → วัตถุอยู่ **ซ้าย** ของกลางภาพ → ป้อมต้องหมุนซ้าย
- error ≈ 0 → วัตถุอยู่กลางภาพพอดีแล้ว → **ยิงได้!**

In [ ]:
h_img, w_img = frame.shape[:2]
cx_frame = w_img // 2

cx, cy, w, h = found_object
error = cx - cx_frame

print(f"กลางภาพอยู่ที่ x = {cx_frame}")
print(f"วัตถุอยู่ที่ x = {cx}")
print(f"error = {error} px  →", "หมุนขวา" if error > 0 else "หมุนซ้าย" if error < 0 else "ตรงกลางพอดี")

# วาดภาพประกอบ: เส้นกลางภาพ (แดง) + จุดวัตถุ (เขียว) + ลูกศรบอก error
error_image = frame.copy()
cv2.line(error_image, (cx_frame, 0), (cx_frame, h_img), (0, 0, 255), 2)
cv2.circle(error_image, (cx, cy), 8, (0, 255, 0), -1)
cv2.arrowedLine(error_image, (cx_frame, cy), (cx, cy), (0, 255, 255), 3)
plt.imshow(cv2.cvtColor(error_image, cv2.COLOR_BGR2RGB))
plt.title(f"เส้นแดง=กลางภาพ | จุดเขียว=วัตถุ | ลูกศร=error ({error:+d}px)")
plt.show()

## 5. จาก error (pixel) เป็นมุมหมุนจริง (องศา) — P-controller ง่ายๆ

จะสั่ง servo ต้องแปลง error หน่วย pixel เป็นหน่วยองศา ใช้สูตรง่ายที่สุดที่ใช้ได้ผลจริง:

```
มุมที่จะหมุน = Kp × error
```

`Kp` (ตัวคูณ) หาได้จากการทดลอง: ถ้าหมุนตามค่าที่คำนวณแล้วเป้ายังไม่ตรงพอ ก็เพิ่ม Kp, ถ้าป้อมส่ายไปมาไม่นิ่งสักที (เกินจุดที่ต้องการทุกครั้ง) ก็ลด Kp

และมี **deadband**: ถ้า error เล็กมากอยู่แล้ว (เช่น <15px) ให้ถือว่า "ตรงพอแล้ว" ไม่ต้องหมุนต่อ (ไม่งั้นป้อมจะขยับกระตุกไม่หยุดไล่ตามค่าความเพี้ยนเล็กๆ น้อยๆ ตลอดเวลา)

In [ ]:
KP = 0.03            # องศาที่หมุนต่อ 1 pixel error (ค่าจริงต้องทดลองปรับ)
DEADBAND_PX = 15      # ถ้า error ไม่เกินนี้ ถือว่าเล็งตรงแล้ว

def how_much_to_turn(error_px):
    if abs(error_px) <= DEADBAND_PX:
        return 0.0, "🎯 ตรงกลางแล้ว ยิงได้!"
    angle = KP * error_px
    direction = "ขวา" if angle > 0 else "ซ้าย"
    return angle, f"หมุน{direction} {abs(angle):.1f}°"

for e in [-200, -50, -10, 0, 8, 60, 300]:
    angle, message = how_much_to_turn(e)
    print(f"error={e:>5} px  →  {message}")

## 🧪 แบบฝึกหัดท้ายบท

โจทย์: เขียนฟังก์ชัน `should_fire(cx, frame_width, deadband=15)` ที่คืนค่า `True` ถ้าเล็งตรงพอจะยิงได้ (error อยู่ในช่วง deadband)
คืนค่า `False` ถ้ายังไม่ตรง

ทดสอบด้วย: `frame_width=640` (กลางภาพคือ 320)
- `should_fire(325, 640)` ควรได้ `True` (ห่างแค่ 5px)
- `should_fire(400, 640)` ควรได้ `False` (ห่างตั้ง 80px)

In [ ]:
def should_fire(cx, frame_width, deadband=15):
    # เขียนโค้ดตรงนี้
    pass

# ทดสอบคำตอบตัวเอง
print(should_fire(325, 640))   # ควรได้ True
print(should_fire(400, 640))   # ควรได้ False

<details><summary>👉 คลิกดูเฉลย</summary>

```python
def should_fire(cx, frame_width, deadband=15):
    error = cx - frame_width // 2
    return abs(error) <= deadband
```
</details>

## 6. เทียบกับโค้ดจริงในโปรเจค

โค้ดที่เพิ่งเขียนทั้งหมดในบทนี้ (หา mask → contour → error → ตัดสินใจ) คือแก่นของไฟล์ `src/aiming.py` จริงในโปรเจค
ฟังก์ชันหลักของมันหน้าตาแบบนี้ (ย่อมาให้อ่านเข้าใจง่าย — ของจริงมี loop คอยหมุนซ้ำจนกว่าจะตรง):

```python
def aim_at(turret, cap, detector, target, on_frame=None):
    while เวลายังไม่หมด:
        ok, frame = cap.read()
        det = detector.detect(frame, target)     # = ขั้นตอน mask + contour ที่เพิ่งทำ
        if det is None:
            turret.pan_by(4)                     # ไม่เจอเป้า -> กวาดหาไปเรื่อยๆ
            continue
        error_px = det.cx - frame.shape[1] / 2    # = error ที่เพิ่งคำนวณ
        if abs(error_px) <= DEADBAND_PX:
            return det                            # ตรงพอแล้ว -> เล็งสำเร็จ กลับไปยิงได้
        step = KP * error_px                      # = สูตร P-controller ที่เพิ่งเขียน
        turret.pan_by(SIGN * step)                # สั่งป้อมหมุนจริง
```

เห็นไหมว่าทุกบรรทัดคือสิ่งที่เพิ่งเรียนมาในบทนี้เป๊ะๆ — ต่างกันแค่ของจริงมันหมุน "จริง" ด้วย servo แทนการพิมพ์ข้อความบอก

---
## ✅ สรุปบทนี้ (สำคัญที่สุดในทั้งคอร์ส)

| เรื่อง | สรุปสั้นๆ |
|---|---|
| `cv2.findContours` | หาเส้นขอบก้อนขาวใน mask |
| `cv2.contourArea` + `max()` | เลือกก้อนใหญ่สุด กรอง noise ทิ้ง |
| `cv2.boundingRect` | ได้กรอบ + จุดกึ่งกลางวัตถุ |
| `error = cx_วัตถุ - cx_กลางภาพ` | ป้อมต้องหมุนไปทางไหน |
| `มุม = Kp × error` + deadband | แปลง error เป็นคำสั่งหมุนจริง |
| **นี่คือ visual servoing** | หมุน → เช็คภาพ → หมุนแก้ ทำซ้ำจนตรง |

➡️ **ไปต่อ:** เปิด `04_distance_measurement.ipynb` — รู้ตำแหน่งซ้าย-ขวาแล้ว คราวนี้มาวัด "ระยะ" กันบ้าง (สำคัญพอกันสำหรับคุมแรงยิง)